In [8]:
import math
import sys
import yaml
sys.path.append('../../python/')  
from periphery import logicGate
from periphery import constant
from periphery.Technology import Technology
from periphery.DFF import DFF
from periphery.adder import Adder
print(constant.INV)

0


In [9]:
with open('../../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

with open('../../mapping.yaml', 'r') as file:
    mapping = yaml.safe_load(file)

with open('../../param.yaml', 'r') as file:
    param = yaml.safe_load(file)

In [21]:
from math import ceil

class ShiftAdd:
    def __init__(self, tech, config, param, clk_freq, num_unit, num_adder_bit, spiking_mode, num_read_pulse):
        self.tech = tech
        self.config = config
        self.param = param
        

        self.clk_freq = clk_freq
        self.feature_size = tech.get_param('featureSize')
        self.validated = self.param['validated']
        self.latency_mode = config['latency_mode']
        self.num_unit = num_unit
        self.numAdder = num_unit
        self.num_adder_bit = num_adder_bit
        self.spiking_mode = spiking_mode
        self.num_read_pulse = num_read_pulse    # the number of add operations (the number of read pulses)

        self.num_dff = None
        self.num_bit_per_dff = None
        self.width_inv_n = None
        self.width_inv_p = None
        self.width_nand_n = None
        self.width_nand_p = None

        self.read_latency = 0
        self.read_dynamic_energy = 0
        self.leakage = 0


        #initialize adder and dff
        self.adder = Adder(num_bit=8,num_adder=1,clk_freq = None,tech=self.tech,config=config,mapping=mapping)
        self.dff = DFF(num_dff=1,param=param,tech=self.tech,clk_freq=1e9,config=config)

        
        if spiking_mode == 'NONSPIKING':        # NONSPIKING: binary format
            self.num_dff = (num_adder_bit + 1 + num_read_pulse - 1) * num_unit  # numAdderBit+1 because the adder output is 1 bit more than the input, and numReadPulse-1 is for shift-and-add extension (shift register)
            self.adder = Adder(num_bit=num_adder_bit,num_adder=num_unit,clk_freq = self.clk_freq,tech=self.tech,config=config,mapping=mapping)
            self.dff = DFF(num_dff=self.num_dff,param=param,tech=self.tech,clk_freq=self.clk_freq,config=config)

        else:   # SPIKING: count spikes
            self.num_bit_per_dff = 2 ** num_adder_bit
            self.num_dff = self.num_bit_per_dff * num_unit
            self.dff = DFF(num_dff=self.num_dff,param=param,tech=self.tech,clk_freq=self.clk_freq,config=config)

        # Currently ignore INV and NAND in shift-add circuit */
	    # PISO shift register (https://en.wikipedia.org/wiki/Shift_register)
	    # INV
        self.width_inv_n = constant.MIN_NMOS_SIZE * self.feature_size
        self.width_inv_p = tech.get_param('pnSizeRatio') * constant.MIN_NMOS_SIZE * self.feature_size
        self.width_nand_n = 2 * constant.MIN_NMOS_SIZE * self.feature_size
        self.width_nand_p = tech.get_param('pnSizeRatio') * constant.MIN_NMOS_SIZE * self.feature_size

        self.initialized = True

    def calculate_area(self, new_height=None, new_width=None, option='NONE'):
        if not self.initialized:
            raise RuntimeError("ShiftAdd must be initialized before area calculation.")

        #to be done: consifer the GAA FET
        h_inv, w_inv, _ = logicGate.calculate_logicgate_area(
            constant.INV, 1,
            self.width_inv_n, self.width_inv_p,
            self.feature_size * constant.MAX_TRANSISTOR_HEIGHT,
            self.tech
        )

        h_nand, w_nand, _ = logicGate.calculate_logicgate_area(
            constant.NAND, 2,
            self.width_nand_n, self.width_nand_p,
            self.feature_size * constant.MAX_TRANSISTOR_HEIGHT,
            self.tech
        )
        ############
        if self.feature_size == 14e-9:
            NEW_CELL_HEIGHT = constant.MAX_TRANSISTOR_HEIGHT_14nm
        elif self.feature_size == 10e-9:
            NEW_CELL_HEIGHT = constant.MAX_TRANSISTOR_HEIGHT_10nm
        elif self.feature_size == 7e-9:
            NEW_CELL_HEIGHT = constant.MAX_TRANSISTOR_HEIGHT_7nm
        elif self.feature_size == 5e-9:
            NEW_CELL_HEIGHT = constant.MAX_TRANSISTOR_HEIGHT_5nm
        elif self.feature_size == 3e-9:
            NEW_CELL_HEIGHT = constant.MAX_TRANSISTOR_HEIGHT_3nm
        elif self.feature_size == 2e-9:
            NEW_CELL_HEIGHT = constant.MAX_TRANSISTOR_HEIGHT_2nm
        elif self.feature_size == 1e-9:
            NEW_CELL_HEIGHT = constant.MAX_TRANSISTOR_HEIGHT_1nm
        else:
            NEW_CELL_HEIGHT = constant.MAX_TRANSISTOR_HEIGHT
        #############
        if new_width and option == 'NONE':
            if self.spiking_mode == 'NONSPIKING':
                adder_area,adder_height,adder_width = self.adder.calculate_area(new_height=None,new_width=None,option='NONE')
                dff_area,dff_height,dff_width = self.dff.calculate_area(new_height=None,new_width=None,option='NONE')

                new_cell_height = self.feature_size * NEW_CELL_HEIGHT
                height = self.adder.height + new_cell_height + self.dff.height
                width = new_width
            else:
                dff_area,dff_height,dff_width = self.dff.calculate_area(new_height=None,new_width=None,option='NONE')
                new_cell_height = self.feature_size * NEW_CELL_HEIGHT
                height = new_cell_height + self.dff.height
                width = new_width
        if new_height and option == 'NONE':
            if self.spiking_mode == 'NONSPIKING':
                adder_area,adder_height,adder_width = self.adder.calculate_area(new_height=None,new_width=None,option='NONE')
                dff_area,dff_height,dff_width = self.dff.calculate_area(new_height=None,new_width=None,option='NONE')
                # Assume the INV and NAND2 are on the same row and the total width of them is smaller than the adder or DFF
                height = new_height
                width = self.adder.width + w_inv + w_nand + self.dff.width
            else:   # SPIKING: count spikes
                dff_area,dff_height,dff_width = self.dff.calculate_area(new_height=None,new_width=None,option='NONE')
                height = new_height
                width = w_inv + w_nand + self.dff.width
        if not new_height and not new_width and option == 'NONE':
            if self.spiking_mode == 'NONSPIKING':
                adder_area,adder_height,adder_width = self.adder.calculate_area(new_height=None,new_width=None,option='NONE')
                dff_area,dff_height,dff_width = self.dff.calculate_area(new_height=None,new_width=None,option='NONE')

                new_cell_height = self.feature_size * NEW_CELL_HEIGHT
                height = self.adder.height + new_cell_height + self.dff.height
                width = self.adder.width + w_inv + w_nand + self.dff.width
            else:
                dff_area,dff_height,dff_width = self.dff.calculate_area(new_height=None,new_width=None,option='NONE')
                new_cell_height = self.feature_size * NEW_CELL_HEIGHT
                height = new_cell_height + self.dff.height
                width = w_inv + w_nand + self.dff.width

        area = width * height
        self.area = area
        self.height = height
        self.width = width
        return area, height, width

    def calculate_latency(self, num_read):
        self.read_latency = 0
        if self.spiking_mode == 'NONSPIKING':
            adder_read_latency = self.adder.calculate_latency(cap_load=self.dff.cap_tg_drain,num_read=1)
            dff_read_latency,dff_write_latency = self.dff.calculate_latency(num_read=1)

            shift_add_latency = adder_read_latency + dff_read_latency

            self.read_latency += shift_add_latency

            if self.latency_mode == 'synchronous':
                self.read_latency = num_read
        else:
            dff_read_latency,dff_write_latency = self.dff.calculate_latency(num_read=self.num_bit_per_dff)
            shift_latency = dff_read_latency

            self.read_latency += shift_latency

            if self.latency_mode == 'synchronous':
                self.read_latency = self.num_bit_per_dff * num_read

        return self.read_latency

    def calculate_power(self, num_read):
        self.leakage = 0
        self.read_dynamic_energy = 0

        if self.spiking_mode == 'NONSPIKING':
            adder_read_energy,adder_leakage = self.adder.calculate_power(num_read=num_read,num_adder_per_op=self.numAdder)
            dff_read_energy,dff_write_energy,dff_leakage = self.dff.calculate_power(num_read=num_read, num_dff_per_op=self.num_dff, validated=self.validated)

            self.read_dynamic_energy += adder_read_energy
            self.read_dynamic_energy += dff_read_energy
            self.leakage += adder_leakage
            self.leakage += dff_leakage
        else:
            dff_read_energy,dff_write_energy,dff_leakage = self.dff.calculate_power(num_read=num_read, num_dff_per_op=self.num_dff, validated=self.validated)
            self.read_dynamic_energy += dff_read_energy
            self.leakage += dff_leakage

        return self.read_dynamic_energy, self.leakage


In [22]:
tech45 = Technology(node_nm=45, roadmap='HP')
# Instantiate and initialize Precharger
pre = ShiftAdd(
    num_adder_bit=4,
    num_unit=1,
    num_read_pulse=1,
    param=param,
    tech=tech45,
    clk_freq=1e9,
    spiking_mode='NONSPIKING',
    # spiking_mode='SPIKING',
    config=config
)

In [23]:
pre_charge_area,pre_charge_height,pre_charge_width = pre.calculate_area(
    new_height=None,
    new_width=None,
    option='NONE'
)
print("Area Result:", pre_charge_area)
print("Height Result:", pre_charge_height)
print("Width Result:", pre_charge_width)

Area Result: 2.0431245600000004e-10
Height Result: 2.8620000000000003e-06
Width Result: 7.1388e-05


In [26]:
read_latency = pre.calculate_latency(
    num_read=1
)
print("Read Latency:", read_latency)
# print("Write Latency:", write_latency)

Read Latency: 6.627753152551134e-10


In [28]:
read_energy,leakage = pre.calculate_power(num_read=1)
print(f"  Read Dynamic Energy: {read_energy:.3e} J")
print(f"  Leakage Power: {leakage:.3e} W")

  Read Dynamic Energy: 3.659e-14 J
  Leakage Power: 1.653e-06 W
